In [7]:
import os
import torch
import warnings
import chunking as C
from tqdm import tqdm
from pathlib import Path
from torch import nn, optim
from resnet20 import resnet20
import torch.nn.functional as F
import json, random, numpy as np
from torchvision import transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR100
from transformer_vae import TVAE, vae_loss
from torch.utils.data import DataLoader, Dataset
from constants import SEED, device, TOKENS_PER_SEQ
from dataset import ResZoo, summon_res_zoo
warnings.filterwarnings("ignore")

### Load the data & Data loaders

- `train_loader`: 27 checkpoints from experts 0–2, ~3,213 sequences.
  Shuffled, `drop_last=True` for consistent batch shapes.
- `validation_loader`: expert 3, never trained on. Used to select the epoch
  and, later, the compression ratio.
- `test_loader`: expert 4, held out entirely. Touched **once**, for the
  final reported generalization number.
- `merge_loader`: the five `ep040` checkpoints. These are experiment
  subjects, not training data, and are excluded from VAE training so the
  encoder cannot have memorized the exact weights we later merge.

At batch size 64 the training set is ~50 steps per epoch, so epochs are cheap
and we can afford 200 of them.

In [6]:
zoo_dir = './zoo_chunks'
training_set, validation_set, test_set, merge_set = summon_res_zoo('./zoo_chunks')
train_dataset = ResZoo(root_dir=zoo_dir, model_ids=training_set)
validation_dataset = ResZoo(root_dir=zoo_dir, model_ids=validation_set)
test_dataset = ResZoo(root_dir=zoo_dir, model_ids=test_set)
merge_dataset = ResZoo(root_dir=zoo_dir, model_ids=merge_set)

print(f'train {len(training_set)}  val {len(validation_set)}  ', f'test {len(test_set)}  merge {len(merge_set)}')

train 27  val 9   test 10  merge 5


In [9]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
validation_loader = DataLoader(validation_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
merge_loader = DataLoader(merge_dataset, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

### Train the TVAE model

## Stage A: Deterministic autoencoder (`r = 1`, KL off)

We train the transformer VAE with **both** of its regularizing knobs disabled:

| Knob | Setting | Meaning |
|---|---|---|
| `latent_dim = 144` | `r = 144/144 = 1` | no bottleneck, the latent is as wide as the input |
| `BETA = 0.0` | KL term off | no pull toward `N(0,1)`, so this is a plain autoencoder |

The point is to give the model every advantage. If it still cannot reconstruct
weights well enough to keep the ResNet functional, the fault is in the
**plumbing**, chunk ordering, denormalization, the mask, device placement,
not in compression or KL regularization. Debugging that is far easier with no
bottleneck in the way.

Compression (`r`) and KL (`beta`) are turned on separately in Stages B and C,
one at a time, so a failure can always be attributed to a single change.

In [10]:
EPOCHS = 200
BETA = 0.0            

tvae_model = TVAE(input_dim=144, model_dim=256, latent_dim=144).to(device)
optimizer = optim.AdamW(tvae_model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [11]:
@torch.no_grad()
def eval_recon(model, loader, beta=0.0):
    '''eval mode makes reparameterize return mu, so reconstruction is deterministic'''
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        chunks = batch['chunks'].to(device, non_blocking=True)
        mask = batch['mask'].to(device, non_blocking=True)
        depth = batch['depth'].to(device, non_blocking=True)
        stage = batch['stage'].to(device, non_blocking=True)
        xhat, mu, logvar = model(chunks, depth, stage)
        _, recon, _ = vae_loss(xhat, chunks, mu, logvar, mask, beta)
        total += recon.item(); n += 1
    return total / max(n, 1)